### Check package exists and install dependency

In [1]:
import importlib
import importlib.metadata

packages = ["numpy", "scipy", "regex", "torch", "transformers", "iopath"]

for pkg in packages:
    try:
        # Try to import the package
        importlib.import_module(pkg)
        
        # Get the version if available
        try:
            version = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            version = "Unknown version"
        
        print(f"{pkg} is installed (version: {version})")
    
    except ModuleNotFoundError:
        print(f"{pkg} is NOT installed")


numpy is installed (version: 2.3.0)
scipy is installed (version: 1.15.3)
regex is installed (version: 2024.11.6)
torch is installed (version: 2.7.1)


/opt/conda/envs/babylm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers is installed (version: 4.52.4)
iopath is installed (version: 0.1.10)


### Run Demo

In [ ]:
import json

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
from parlai.zoo.blender.blender_3B import download
from parlai.core.opt import Opt

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot/
from controllable_blender import ControllableBlender

agent_opt = json.load(open("blender_3B.opt", 'r'))

# Set to "vocab" for vocabulary restriction, "rerank" for candidate reranking
agent_opt["inference"] = "rerank"
# Same top-k sampling configs for all settings described in the paper
agent_opt["beam_size"] = 20
agent_opt["topk"] = 40

# Settings for rerank methods (not used if "inference" == "vocab")
agent_opt["rerank_cefr"] = "B2"                       # CEFR level to adjust reranking. Possible values: ['A2', 'B1', 'B2', 'C1', 'C2'].
agent_opt["rerank_tokenizer"] = "distilroberta-base"        # Tokenizer from Huggingface Transformers. Must be compatible with "rerank_model"
agent_opt["rerank_model"] = "complexity_model"              # Model fine-tuned on complexity data
agent_opt["rerank_model_device"] = "cuda"                   # Device for complexity model
agent_opt["penalty_stddev"] = 2                             # Controls how harshly sub-tokens are penalised (lower = harsher). Use -1 to remove penalties
agent_opt["filter_path"] = "data/filter.txt"                # Path to list of English words to ensure OOV words are not generated. Capitalised words are ignored. Use empty string to remove filter

# Settings for vocab methods (not used if "inference" == "rerank")
agent_opt["wordlist_path"] = "data/sample_wordlist.txt"          # Path to list of vocab the chatbot is restricted to


download(agent_opt["datapath"])

agent = ControllableBlender(agent_opt)

#import baby model
from transformers import AutoTokenizer, LlamaForCausalLM
from parlai.core.agents import Agent
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

#instantiate your model here
class TransformersAgentWrapper(Agent):
    def __init__(self, opt, model_name="timinar/baby-llama-58m", max_new_tokens=60):
        super().__init__(opt)
        self.id = "BabyLLaMA"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load tokenizer + model
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            weights_only=False
        ).to(self.device)

        self.max_new_tokens = max_new_tokens
        self.context = ""

    def observe(self, observation):
        if 'episode_done' not in observation:
            observation = observation.copy()  # avoid mutating input dict
            observation['episode_done'] = False
        self.observation = observation
        if not observation.get("episode_done", False):
            self.context += f"\n{observation.get('text', '')}"

    def act(self):
        inputs = self.tokenizer(
            self.context,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            pad_token_id=self.tokenizer.pad_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8
        )

        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        reply = full_text[len(self.context):].strip() or "[No response]"
        self.context += f"\n{reply}"
        return {"id": self.id, "text": reply}

from parlai.core.params import ParlaiParser
from parlai.core.worlds import DialogPartnerWorld
from controllable_blender import ControllableBlender

class TwoBotDialogPartnerWorld(DialogPartnerWorld):
    def __init__(self, opt, agent1, agent2, max_turns=5):
        super().__init__(opt, [agent1, agent2])
        self.turns = 0
        self.max_turns = max_turns

    def parley(self):
        acts = []
        for idx, agent in enumerate(self.agents):
            # First agent's turn
            if idx == 0 and self.turns == 0:
                text = "The teacher greets you. Respond as the student."
            else:
                # Use the last message from the other agent
                text = acts[-1]['text'] if acts else ""

            obs = {"text": text, "episode_done": False}
            agent.observe(obs)
            acts.append(agent.act())

        self.acts = acts
        for m in acts:
            print(f"{m['id']}: {m.get('text', '')}")

        self.turns += 1

    def episode_done(self):
        return self.turns >= self.max_turns


if __name__ == "__main__":
    parser = ParlaiParser()
    opt = parser.parse_args([])

    # Instantiate agents
    hf_agent = TransformersAgentWrapper(opt, model_name="timinar/baby-llama-58m")
    parlai_agent = agent

    # Create and run world
    world = TwoBotDialogPartnerWorld(opt, hf_agent, parlai_agent, max_turns=5)

    while not world.episode_done():
        world.parley()

/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot
10:32:39 INFO | Using CUDA
10:32:39 INFO | loading dictionary from ParlAI/data/models/blender/blender_3B/model.dict
10:32:39 INFO | num words = 8008
10:33:13 INFO | Total parameters: 2,696,268,800 (2,695,613,440 trainable)
10:33:13 INFO | Loading existing model params from ParlAI/data/models/blender/blender_3B/model
BabyLLaMA: I had to go up to the school.
- How are you?
- I'm just trying to do it.
- What?
- I'm just gonna be here.
- That's okay, you have to go.
You're not gonna be with me again.
ControllableBlender: That's very sad. Why are you not going to be with him again? The relationship is over?
BabyLLaMA: I'm going to be with you, and I don't want you to ever have to do it.
Well, I thought you'd like to.
- What?
- I have a great life.
I want you to go out and join me.
Do you want to com

This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (256). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


BabyLLaMA: - You've missed a lot.
- I know.
- It's very nice of course you.
- Yeah.
- Yeah.
- Yeah. 
I'm.
- I'm a little.
I'm. 
- I want to give me a couple
ControllableBlender: That would be nice. How many kids do you have?  I don't. I have no kids.
BabyLLaMA: - for a little bit.
- I just to be a little bit of course.
- I'll be happy.
- I know, uh, what?
- You know what?
- You and I like the way you.
- I'm not.
- I'm
ControllableBlender: Do you have any pets? I have two dogs. One of them is the best. 


### Simplified parlai agent interaction

In [1]:
import json

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
from parlai.zoo.blender.blender_3B import download
from parlai.core.opt import Opt

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot/
from controllable_blender import ControllableBlender

agent_opt = json.load(open("blender_3B.opt", 'r'))


# Set to "vocab" for vocabulary restriction, "rerank" for candidate reranking
agent_opt["inference"] = "rerank"
# Same top-k sampling configs for all settings described in the paper
agent_opt["beam_size"] = 20
agent_opt["topk"] = 40

# Settings for rerank methods (not used if "inference" == "vocab")
agent_opt["rerank_cefr"] = "B2"                       # CEFR level to adjust reranking. Possible values: ['A2', 'B1', 'B2', 'C1', 'C2'].
agent_opt["rerank_tokenizer"] = "distilroberta-base"        # Tokenizer from Huggingface Transformers. Must be compatible with "rerank_model"
agent_opt["rerank_model"] = "complexity_model"              # Model fine-tuned on complexity data
agent_opt["rerank_model_device"] = "cuda"                   # Device for complexity model
agent_opt["penalty_stddev"] = 2                             # Controls how harshly sub-tokens are penalised (lower = harsher). Use -1 to remove penalties
agent_opt["filter_path"] = "data/filter.txt"                # Path to list of English words to ensure OOV words are not generated. Capitalised words are ignored. Use empty string to remove filter

# Settings for vocab methods (not used if "inference" == "rerank")
agent_opt["wordlist_path"] = "data/sample_wordlist.txt"          # Path to list of vocab the chatbot is restricted to


download(agent_opt["datapath"])

agent = ControllableBlender(agent_opt)

/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot


/opt/conda/envs/babylm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


12:42:33 INFO | Using CUDA
12:42:33 INFO | loading dictionary from ParlAI/data/models/blender/blender_3B/model.dict
12:42:34 INFO | num words = 8008
12:43:07 INFO | Total parameters: 2,696,268,800 (2,695,613,440 trainable)
12:43:07 INFO | Loading existing model params from ParlAI/data/models/blender/blender_3B/model


In [7]:
text = "B::And nothing is being done about it. Uh, the laws exist and are frequently upheld in, in, uh, in Appeals Court just because of technicalities and because of maybe small little holes that their defending attorney can find. And it's, it's really getting out of hand in many states. A::Well, the term technicality . The law enforcement community, uh, uh, you know, has to, has to separate the difference between somebody who is being set up in which, uh, grievous acts are done to, uh, to, you know, to get somebody into a, a situation where they're going to be guilty of, of a crime. Or whether, uh, and whether the rights of that individual are been, have been, you know, impuned. Uh, but or whether there's just, you know, a policeman has just made a, uh, a, you know, a non, a noncritical error, though be it not the right way to do it but, but, you know, the, the merits of the case in terms of, you know, the guy was a law breaker, as being supportive. Now, I, I'm, at this juncture I, you know, I'm, I'm not sure, you know, what constitutes a, a technicality. You know, that, that's what all these, these hearings are about and that's what all these, you know, court cases are about. I mean our, uh, our, our glorious, uh, you know, mayor here in Washington is six days away from getting out of, out of the can and, uh, you know, he, he tried to appeal his conviction. Uh, and, you know, it didn't work. But be that as it may, everybody who got enough money will pump the appeal process dry. Uh, in, in the old days, you know, and say round about times of battle of Hastings, you know, and the villages if you were a transgressor, they, they either, you know, drove you out in the woods or you became a ward of somebody and he, you were his slave. And if he didn't like what you did, he killed you. And that has, that's pretty effective. Uh, you know, it's not good for civil rights, I guess, but it's pretty effective in that, you know, you've got to get along in the community and if you don't you'll perish. Either by the hand of your, your, your master or by being pushed out in the woods. So, I, I, I mean as, as man has gotten more complicated so all of the, uh, imaginations to, uh, you know, protect him from, from being, uh, dumped on by, uh, civilian authority in, in in criminal actions, especially, you know, murder cases and that sort of thing.\n"

agent.reset() # optional between turns
obs = {'text': text, 'episode_done': False}
agent.observe(obs)
reply = agent.act()
print(reply.get('text', ''))

Yeah, that doesn't sound right, but I don't know enough about murder cases to dispute it.


### Batch interact experiments

In [1]:
import json

with open("/workspace/babylm-interaction/TnD_reformed/experiments/orpo_training/final_iterative_training_3b_data_babylm_opt_seqlen_1024_stable_config.json", "r") as f:
    data = json.load(f)

from parlai.core.message import Message
obs = [Message({'text': data[i]['prompt'], 'episode_done': False}) for i in range(15,30)]

In [2]:
# rerank option
import json

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
from parlai.zoo.blender.blender_3B import download
from parlai.core.opt import Opt

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot/
from controllable_blender import ControllableBlender

agent_opt = json.load(open("blender_3B.opt", 'r'))


# Set to "vocab" for vocabulary restriction, "rerank" for candidate reranking
agent_opt["inference"] = "rerank"
# Same top-k sampling configs for all settings described in the paper
agent_opt["beam_size"] = 20
agent_opt["topk"] = 40

# Settings for rerank methods (not used if "inference" == "vocab")
agent_opt["rerank_cefr"] = "B2"                       # CEFR level to adjust reranking. Possible values: ['A2', 'B1', 'B2', 'C1', 'C2'].
agent_opt["rerank_tokenizer"] = "distilroberta-base"        # Tokenizer from Huggingface Transformers. Must be compatible with "rerank_model"
agent_opt["rerank_model"] = "complexity_model"              # Model fine-tuned on complexity data
agent_opt["rerank_model_device"] = "cuda"                   # Device for complexity model
agent_opt["penalty_stddev"] = 2                             # Controls how harshly sub-tokens are penalised (lower = harsher). Use -1 to remove penalties
agent_opt["filter_path"] = "data/filter.txt"                # Path to list of English words to ensure OOV words are not generated. Capitalised words are ignored. Use empty string to remove filter

# Settings for vocab methods (not used if "inference" == "rerank")
agent_opt["wordlist_path"] = "data/sample_wordlist.txt"          # Path to list of vocab the chatbot is restricted to


download(agent_opt["datapath"])

agent = ControllableBlender(agent_opt)

/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot


/opt/conda/envs/babylm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


13:18:24 INFO | Using CUDA
13:18:24 INFO | loading dictionary from ParlAI/data/models/blender/blender_3B/model.dict
13:18:24 INFO | num words = 8008
13:18:58 INFO | Total parameters: 2,696,268,800 (2,695,613,440 trainable)
13:18:58 INFO | Loading existing model params from ParlAI/data/models/blender/blender_3B/model


In [3]:
agent.reset()
agent.batch_respond(obs)

["I've never been there, but I have been wanting to go.  Have you ever been?",
 "What do you have to sell? I don't have much but I would like to have some things.",
 'That does sound troublesome. Do you have an escape plan in place just in case?',
 "I can see what you mean.  How important is Guatamala's national history to people?",
 "What kind of work are you doing? I work in an office, so I don't have much experience with aggressive women.",
 "My wife and I used my wife's money to pay bills.  I didn't even see the money.",
 'Yeah, I know what you mean. I have some friends that are really into classical and just listen to it all the time.',
 'What kind of car do you have? I have a Chevy Silverado. It is my first pick.',
 'Oh, I know.  I went through jury selection a couple years ago at work.',
 'I would be very grateful. I hate being unemployed. It is hard out there.',
 'Wow.  Yeah, I know what you mean!  People can be really rude when they ask about your high school!',
 "Well we shou

In [7]:
!nvidia-smi

Tue Aug 12 13:20:54 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.08             Driver Version: 535.161.08   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:1A:00.0 Off |                    0 |
| N/A   41C    P0             117W / 700W |  26329MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [4]:
# with less beam size option
import json

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
from parlai.zoo.blender.blender_3B import download
from parlai.core.opt import Opt

%cd /workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot/
from controllable_blender import ControllableBlender

agent_opt = json.load(open("blender_3B.opt", 'r'))


# Set to "vocab" for vocabulary restriction, "rerank" for candidate reranking
agent_opt["inference"] = "rerank"
# Lesser beam to increase speed
agent_opt["beam_size"] = 4             
agent_opt["topk"] = 40
agent_opt["topp"] = 0.9
agent_opt["temperature"] = 0.7

# Settings for rerank methods (not used if "inference" == "vocab")
agent_opt["rerank_cefr"] = "B2"                       # CEFR level to adjust reranking. Possible values: ['A2', 'B1', 'B2', 'C1', 'C2'].
agent_opt["rerank_tokenizer"] = "distilroberta-base"        # Tokenizer from Huggingface Transformers. Must be compatible with "rerank_model"
agent_opt["rerank_model"] = "complexity_model"              # Model fine-tuned on complexity data
agent_opt["rerank_model_device"] = "cuda"                   # Device for complexity model
agent_opt["penalty_stddev"] = 2                             # Controls how harshly sub-tokens are penalised (lower = harsher). Use -1 to remove penalties
agent_opt["filter_path"] = "data/filter.txt"                # Path to list of English words to ensure OOV words are not generated. Capitalised words are ignored. Use empty string to remove filter

# Settings for vocab methods (not used if "inference" == "rerank")
agent_opt["wordlist_path"] = "data/sample_wordlist.txt"          # Path to list of vocab the chatbot is restricted to


download(agent_opt["datapath"])

agent = ControllableBlender(agent_opt)

/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ParlAI
/workspace/babylm-interaction/TnD_reformed/experiments/interactive_experiments/ControllableComplexityChatbot
13:19:58 INFO | Using CUDA
13:19:58 INFO | loading dictionary from ParlAI/data/models/blender/blender_3B/model.dict
13:19:58 INFO | num words = 8008
13:20:29 INFO | Total parameters: 2,696,268,800 (2,695,613,440 trainable)
13:20:29 INFO | Loading existing model params from ParlAI/data/models/blender/blender_3B/model


In [5]:
agent.reset()
agent.batch_respond(obs)

["That's too bad. I have been to the capital city, San Jose, and it's a wonderful city.",
 "Yeah, it's a big thing. I have to save up for a house. It's crazy to think about.",
 'Oh wow. That makes sense. I can definitely see how that would be a big deal. ',
 "That's really nice.  I like the idea of a church having a leader.  It's nice to have someone to look up to.",
 'I agree with you. I think it has a lot to do with the fact that women are socially and emotionally different than men.',
 'I bet you are going to regret that after you are done with school.  Have you thought about transferring?',
 "That's great. I like all sorts of music. I even enjoy listening to it in the bath! ",
 'What is something you like to do for fun? I have never done auto repair myself. B',
 'Jury nullification allows people to vote not guilty. People have the right to not give a damn about the law.',
 'I get where you are coming from, but the fact is,  they are not employees, they are independent contractors.'

In [6]:
!nvidia-smi

Tue Aug 12 13:20:53 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.08             Driver Version: 535.161.08   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:1A:00.0 Off |                    0 |
| N/A   41C    P0             117W / 700W |  26329MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [8]:
obs = [Message({'text': data[i]['prompt'], 'episode_done': False}) for i in range(30,45)]
agent.reset()
agent.batch_respond(obs)

["So you're going into public service?  Me too!  What do you want to do?",
 'I am in the same boat.  I think we both need to commit to something and make it a habit.  Good luck!',
 "I've never been a good hunter, but I can see myself getting into it. I love to go target shooting. ",
 "That does sound troublesome. I'm from the UK so I've never had to deal with that.",
 "Well, it's important to make sure you're doing what you can to protect your privacy.",
 "I'm sure you can find a mechanic that will do it for cheaper. I'm glad you found someone that can do it.",
 "Oh no, I'm sorry you had to deal with that. I can't imagine having a kid and having to wake up so early.",
 'My name is Rob and I make $100,000 a year. It is a very nice place.',
 "Well, we've got a lot of money going to research for cancer, but not so much for preventative care. ",
 "That's great! I love when a team starts to do well! It's so exciting for everyone involved.",
 "Wow.  Just wow.  I can't even imagine having a 4

In [10]:
!nvidia-smi

Tue Aug 12 13:21:32 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.08             Driver Version: 535.161.08   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:1A:00.0 Off |                    0 |
| N/A   41C    P0             117W / 700W |  26329MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--